In [1]:
import numpy as np
from glob import glob
from DNN.lw_spacenet import SeismicDataset, SeismicTrainer, create_data_loaders, UNet3D
from util.plotting import plot_3d_array_interactive

In [2]:
def slice_data_chunks_with_stride(data, chunk_size=(128, 128, 128), stride=None):
    """
    Slice input data of shape (150, 150, 2000) into overlapping chunks of specified size.
    
    Args:
        data: Input numpy array of shape (150, 150, 2000)
        chunk_size: Tuple of (height, width, depth) for each chunk
        stride: Step size between chunks (default is half the chunk size for 50% overlap)
    
    Returns:
        List of numpy arrays, each of shape chunk_size
    """
    h, w, d = data.shape
    ch, cw, cd = chunk_size
    
    # Default stride is half the chunk size for 50% overlap
    if stride is None:
        stride = (ch // 2, cw // 2, cd // 2)
    
    sh, sw, sd = stride
    
    chunks = []
    
    # Calculate how many chunks we can fit in each dimension with the given stride
    h_steps = max(1, (h - ch) // sh + 1)
    w_steps = max(1, (w - cw) // sw + 1)
    d_steps = max(1, (d - cd) // sd + 1)
    
    for i in range(h_steps):
        for j in range(w_steps):
            for k in range(d_steps):
                start_h, end_h = i * sh, i * sh + ch
                start_w, end_w = j * sw, j * sw + cw
                start_d, end_d = k * sd, k * sd + cd
                
                # Ensure we don't exceed the data bounds
                end_h = min(end_h, h)
                end_w = min(end_w, w)
                end_d = min(end_d, d)
                
                # Adjust start positions if we're near the boundary
                start_h = max(0, end_h - ch)
                start_w = max(0, end_w - cw)
                start_d = max(0, end_d - cd)
                
                chunk = data[start_h:end_h, start_w:end_w, start_d:end_d]
                chunks.append(chunk)
    del chunks[0:5]
    return np.array(chunks)

def normalize_seismic(seismic):
    """Normalize seismic to [0, 1] using min-max."""
    min_val = np.min(seismic)
    max_val = np.max(seismic)
    if max_val - min_val == 0:
        return np.zeros_like(seismic)
    return (seismic - min_val) / (max_val - min_val)

def normalize_rgt(rgt):
    """Normalize RGT by mean and std."""
    mean_val = np.mean(rgt)
    std_val = np.std(rgt)
    if std_val == 0:
        return np.zeros_like(rgt)
    normalized = (rgt - mean_val) / std_val
    return normalized

In [3]:
seismic = glob("../data/synthetic_data/run/*150-150-2000*/seismicCubes_cumsum_fullstack*.npy")
age = glob("../data/synthetic_data/run/*150-150-2000*/faulted_age*.npy")
seis_chunk_norm = []
age_chunk = []
for s, a in zip(seismic, age):
    sdf = np.load(s)
    sdf_norm = normalize_seismic(sdf)
    adf = np.load(a)
    seis_chunk_norm.extend(slice_data_chunks_with_stride(sdf_norm))
    age_chunk.extend(slice_data_chunks_with_stride(adf))
age_chunk_norm = []
for chunk in age_chunk:
    age_chunk_norm.append(normalize_rgt(chunk))

In [ ]:
import datetime
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
log_dir = f'../data/DNN models/.logs/run_{timestamp}'

dataset = SeismicDataset(seis_chunk_norm, age_chunk_norm, augm=False)
train_loader, val_loader = create_data_loaders(dataset, batch_size=3)
model = UNet3D(in_channels=1, out_channels=1, init_features=16)
trainer = SeismicTrainer(model, train_loader, val_loader, log_dir=log_dir, tv_loss_weight=0.01)

/home/spaceswimmer/miniconda3/envs/diploma/lib/python3.12/site-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


In [5]:
trainer.train(num_epochs=20, save_dir=f"../data/DNN models/run-{timestamp}")


Epoch 1/20
------------------------------
Batch 0/20, Loss: 1.015625
Batch 5/20, Loss: 0.906250
Batch 10/20, Loss: 0.734375
Batch 15/20, Loss: 0.679688
Train Loss: 0.791602
Val Loss: 0.612500
Saved best model with val_loss: 0.612500

Epoch 2/20
------------------------------
Batch 0/20, Loss: 0.593750
Batch 5/20, Loss: 0.527344
Batch 10/20, Loss: 0.496094
Batch 15/20, Loss: 0.496094
Train Loss: 0.514941
Val Loss: 0.423438
Saved best model with val_loss: 0.423438

Epoch 3/20
------------------------------
Batch 0/20, Loss: 0.447266
Batch 5/20, Loss: 0.380859
Batch 10/20, Loss: 0.365234
Batch 15/20, Loss: 0.337891
Train Loss: 0.372168
Val Loss: 0.317578
Saved best model with val_loss: 0.317578

Epoch 4/20
------------------------------
Batch 0/20, Loss: 0.298828
Batch 5/20, Loss: 0.287109
Batch 10/20, Loss: 0.302734
Batch 15/20, Loss: 0.318359
Train Loss: 0.266357
Val Loss: 0.241992
Saved best model with val_loss: 0.241992

Epoch 5/20
------------------------------
Batch 0/20, Loss: 0.2

([0.7916015625,
  0.51494140625,
  0.37216796875,
  0.266357421875,
  0.1787109375,
  0.1404052734375,
  0.1156005859375,
  0.1030517578125,
  0.0820068359375,
  0.07381591796875,
  0.0722412109375,
  0.06783447265625,
  0.06400146484375,
  0.06015625,
  0.05906982421875,
  0.055712890625,
  0.05537109375,
  0.05740966796875,
  0.05340576171875,
  0.04970703125],
 [0.6125,
  0.4234375,
  0.317578125,
  0.2419921875,
  0.1751953125,
  0.1529296875,
  0.1353515625,
  0.11298828125,
  0.10419921875,
  0.10341796875,
  0.0974609375,
  0.097265625,
  0.0904296875,
  0.08740234375,
  0.08134765625,
  0.085546875,
  0.08154296875,
  0.07998046875,
  0.0755859375,
  0.071923828125])

In [20]:
dataset[0][0].shape

torch.Size([1, 128, 128, 128])

In [46]:
seis, rgt = dataset[250]
seis, rgt = seis.float(), rgt.float()
plot_3d_array_interactive(seis[0], axis='x')
plot_3d_array_interactive(rgt[0], axis='x', cmap='viridis')

interactive(children=(IntSlider(value=0, description='X-index:', max=127), Output()), _dom_classes=('widget-in…

interactive(children=(IntSlider(value=0, description='X-index:', max=127), Output()), _dom_classes=('widget-in…

In [ ]:
# seis, rgt = dataset[3]
# seis, rgt = seis.float(), rgt.float()
# plot_3d_array_interactive(seis[0], axis='x')
# plot_3d_array_interactive(rgt[0], axis='x', cmap='viridis')

interactive(children=(IntSlider(value=0, description='X-index:', max=127), Output()), _dom_classes=('widget-in…

interactive(children=(IntSlider(value=0, description='X-index:', max=127), Output()), _dom_classes=('widget-in…